# Feature Engineering for Fraud Detection

This notebook covers feature engineering techniques for the IEEE Fraud Detection dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load optimized data
with open('../../data/train_optimized.pkl', 'rb') as f:
    df = pickle.load(f)

print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
df = df.set_index('TransactionID')
df.head()

## 1. V Features Correlation Analysis (>95% Correlation Drop)

In [ ]:
# V features correlation analysis
v_cols = [col for col in df.columns if col.startswith('V')]
print(f"Total V features: {len(v_cols)}")

# Calculate correlation matrix for V features
v_corr_matrix = df[v_cols].corr().abs()

# Find highly correlated pairs (>95%)
high_corr_pairs = []
features_to_drop = []

# Get upper triangle indices (to avoid duplicates)
upper_tri = np.triu(np.ones(v_corr_matrix.shape), k=1).astype(bool)

for i in range(len(v_corr_matrix.columns)):
    for j in range(i+1, len(v_corr_matrix.columns)):
        if v_corr_matrix.iloc[i, j] > 0.95:
            col1 = v_corr_matrix.columns[i]
            col2 = v_corr_matrix.columns[j]
            correlation = v_corr_matrix.iloc[i, j]
            high_corr_pairs.append((col1, col2, correlation))
            # Drop the second feature (col2)
            if col2 not in features_to_drop:
                features_to_drop.append(col2)

print(f"\nFound {len(high_corr_pairs)} pairs with >95% correlation")
print(f"Features to drop: {len(features_to_drop)}")

# Display some examples
print(f"\nSample highly correlated pairs:")
for pair in high_corr_pairs[:10]:
    print(f"  {pair[0]} <-> {pair[1]}: {pair[2]:.4f}")

In [ ]:
# Visualize correlation matrix (sampled for readability)
sample_v_cols = v_cols[:30]  # First 30 V features
plt.figure(figsize=(14, 12))
sns.heatmap(df[sample_v_cols].corr(), cmap='coolwarm', center=0, 
            xticklabels=True, yticklabels=True)
plt.title('V Features Correlation Matrix (First 30 Features)')
plt.tight_layout()
plt.show()

print(f"\n✅ Features to drop (>95% correlation):")
print(features_to_drop[:20])  # Show first 20

## 2. Time-Based Feature Engineering

In [ ]:
# TransactionDT is in seconds from a reference point
# Create simple time features (common in Kaggle competitions)

# Basic time features
df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day_of_week'] = (df['TransactionDT'] // (3600 * 24)) % 7
df['day'] = (df['TransactionDT'] // (3600 * 24)) % 30

# Additional useful features
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)

print("Created time features:")
print("  - hour: Hour of the day (0-23)")
print("  - day_of_week: Day of week (0-6)")
print("  - day: Day of month (0-29)")
print("  - is_weekend: Weekend flag (0/1)")
print("  - is_night: Night time flag (0/1)")

# Show distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Hour distribution by fraud
for fraud_val, color, label in [(0, '#2ecc71', 'Not Fraud'), (1, '#e74c3c', 'Fraud')]:
    data = df[df['isFraud'] == fraud_val]['hour']
    axes[0, 0].hist(data, bins=24, alpha=0.6, color=color, label=label, density=True)
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Transaction Hour Distribution')
axes[0, 0].legend()

# Weekend fraud rate
weekend_fraud = df.groupby('is_weekend')['isFraud'].mean() * 100
axes[0, 1].bar(['Weekday', 'Weekend'], weekend_fraud.values, color=['#3498db', '#e74c3c'])
axes[0, 1].set_ylabel('Fraud Rate (%)')
axes[0, 1].set_title('Fraud Rate: Weekday vs Weekend')

# Night fraud rate
night_fraud = df.groupby('is_night')['isFraud'].mean() * 100
axes[1, 0].bar(['Day', 'Night'], night_fraud.values, color=['#3498db', '#e74c3c'])
axes[1, 0].set_ylabel('Fraud Rate (%)')
axes[1, 0].set_title('Fraud Rate: Day vs Night')

# Day of week fraud rate
dow_fraud = df.groupby('day_of_week')['isFraud'].mean() * 100
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1, 1].bar(days, dow_fraud.values, color='#9b59b6')
axes[1, 1].set_ylabel('Fraud Rate (%)')
axes[1, 1].set_title('Fraud Rate by Day of Week')

plt.tight_layout()
plt.show()

## 3. Categorical Feature Encoding

In [ ]:
# Identify categorical columns
categorical_cols = [col for col in df.columns if df[col].dtype in ['object', 'category']]
print(f"Categorical columns: {len(categorical_cols)}")

# High cardinality vs low cardinality
high_cardinality = [col for col in categorical_cols if df[col].nunique() > 50]
low_cardinality = [col for col in categorical_cols if df[col].nunique() <= 50]

print(f"\nLow cardinality (<=50 unique): {len(low_cardinality)}")
print(f"High cardinality (>50 unique): {len(high_cardinality)}")

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a copy for encoding
df_encoded = df.copy()

# Label encoding for all categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    # Handle NaN values
    df_encoded[col] = df_encoded[col].fillna('MISSING')
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

print(f"Encoded {len(categorical_cols)} categorical columns")
print(f"\nSample encoded values:")
for col in categorical_cols[:5]:
    print(f"  {col}: {df_encoded[col].nunique()} unique values")

In [ ]:
# Frequency encoding for high cardinality features
for col in high_cardinality:
    freq = df[col].value_counts(normalize=True)
    df_encoded[f'{col}_freq'] = df[col].map(freq).fillna(0)

print(f"Created {len(high_cardinality)} frequency encoded features")
print("New features:", [f'{col}_freq' for col in high_cardinality[:5]])

## 4. Transaction Amount Features

In [ ]:
# Transaction amount transformations
df_encoded['TransactionAmt_log'] = np.log1p(df_encoded['TransactionAmt'])
df_encoded['TransactionAmt_decimal'] = df_encoded['TransactionAmt'] - df_encoded['TransactionAmt'].astype(int)

# Binned transaction amount
bins = [0, 50, 100, 200, 500, 1000, 5000, float('inf')]
labels = list(range(len(bins)-1))
df_encoded['TransactionAmt_bin'] = pd.cut(df_encoded['TransactionAmt'], bins=bins, labels=labels).astype(float)

print("Created transaction amount features:")
print("  - TransactionAmt_log: Log transformed amount")
print("  - TransactionAmt_decimal: Decimal part of amount")
print("  - TransactionAmt_bin: Binned amount (0-6)")

## 5. Prepare Final Dataset

In [ ]:
# Drop highly correlated V features
print(f"Before dropping: {df_encoded.shape[1]} features")

# Remove features that exist in our drop list
cols_to_drop = [col for col in features_to_drop if col in df_encoded.columns]
df_encoded = df_encoded.drop(columns=cols_to_drop)

print(f"After dropping {len(cols_to_drop)} highly correlated features: {df_encoded.shape[1]} features")

In [ ]:
# Fill remaining NaN values
numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns

# Fill with median for numeric columns
for col in numeric_cols:
    if df_encoded[col].isnull().sum() > 0:
        df_encoded[col] = df_encoded[col].fillna(df_encoded[col].median())

print(f"Remaining NaN values: {df_encoded.isnull().sum().sum()}")
print(f"\nFinal dataset shape: {df_encoded.shape}")

## 6. Quick Model Test (LightGBM)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import lightgbm as lgb

# Prepare features and target
X = df_encoded.drop(columns=['isFraud'])
y = df_encoded['isFraud']

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Fraud rate - Train: {y_train.mean()*100:.2f}%, Val: {y_val.mean()*100:.2f}%")

In [ ]:
# LightGBM model
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbosity': -1,
    'is_unbalance': True  # Handle imbalanced data
}

model = lgb.LGBMClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

In [ ]:
# Predictions and evaluation
y_pred_proba = model.predict_proba(X_val)[:, 1]
y_pred = model.predict(X_val)

auc_score = roc_auc_score(y_val, y_pred_proba)
print(f"\n{'='*60}")
print(f"VALIDATION RESULTS")
print(f"{'='*60}")
print(f"\nROC-AUC Score: {auc_score:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_val, y_pred))

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 10))
plt.barh(importance['feature'].head(30), importance['importance'].head(30), color='#3498db')
plt.xlabel('Importance')
plt.title('Top 30 Feature Importances (LightGBM)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 most important features:")
print(importance.head(10).to_string(index=False))

In [ ]:
# Save processed data
print(f"\n{'='*60}")
print("FEATURE ENGINEERING SUMMARY")
print(f"{'='*60}")
print(f"""
✅ Completed Feature Engineering:

1. V Features Correlation Drop:
   - Dropped {len(cols_to_drop)} features with >95% correlation

2. Time Features Created:
   - hour, day_of_week, day, is_weekend, is_night

3. Categorical Encoding:
   - Label encoding for all categorical columns
   - Frequency encoding for high cardinality columns

4. Amount Features:
   - Log transformed, decimal part, binned

5. Model Performance:
   - LightGBM ROC-AUC: {auc_score:.4f}

Final Dataset: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} features
""")
print(f"{'='*60}")

In [ ]:
# Optional: Save processed dataset
# with open('../../data/train_featured.pkl', 'wb') as f:
#     pickle.dump(df_encoded, f)
# print("Processed data saved to train_featured.pkl")